# Lots of match-ups
**Author:** Eli Holmes (NOAA)</br>

When we have 10s of thousands of points for matchups, we need to be careful with memory. Just opening the meta-data for all the netcdfs, hundreds of files, will eat up a lot of memory, slowly but surely.

**What do we do?**

1. When we open the files, we create `ds` with `chunks{}` so that dask knows the chunk structure and will not try to load the whole file.
2. We compute the vector of indices for the Argo points and use `ds.vindex([vector_i, vector_j)`. That will load only the needed chunks.
3. We work through the files in batches rather than one massive for loop. This will keep the memory from exploding on us as we open files.


## Load points

This is 15k+ of BGC-Argo points.

In [1]:
import pandas as pd
df = pd.read_parquet("https://raw.githubusercontent.com/fish-pace/fish-pace-datasets/main/datasets/chla_z/data/CHLA_argo_profiles.parquet")
df_points = df[["TIME", "LATITUDE", "LONGITUDE"]].rename(
    columns={
        "TIME": "time",
        "LATITUDE": "lat",
        "LONGITUDE": "lon"
    }
)
print(len(df_points))
df_points.head()

15833


,time,lat,lon
0,2024-03-01 21:23:16.002000128,54.6582,-19.2434
1,2024-03-11 20:45:53.002000128,54.9187,-18.9609
2,2024-03-21 21:21:39.002000128,55.2967,-18.8331
3,2024-03-31 21:31:53.002000128,55.7268,-18.8653
4,2024-03-07 18:01:17.002000128,17.6665,-46.0155


## We create a plan for these points

15k points takes about 15 seconds to search EarthData catalog and develop a plan.

In [2]:
%%time
import point_collocation as pc
plan = pc.plan(
    df_points,
    data_source="earthaccess",
    source_kwargs={
        "short_name": "PACE_OCI_L3M_RRS",
    }
)

CPU times: user 2.63 s, sys: 102 ms, total: 2.73 s
Wall time: 7 s


## Now matchup

This explodes memory.

In [3]:
%%time
res = pc.matchup(plan, geometry="grid", variables=["Rrs"], save_dir="_temp_data")

CPU times: user 9 μs, sys: 1 μs, total: 10 μs
Wall time: 11.2 μs


TypeError: matchup() got an unexpected keyword argument 'save_dir'

# Workflow to process one PACE file

Assume:

* The files are for a regular grid, i.e. level 3 data
* The files do not have a time coordinate. This is how the PACE level 3 daily and 8-day files are currently structured.
* We are looking at a variable, like Rrs, that has a 3rd dimension of wavelength or similar. Usually called wavelength or wavelength_3d. So the dims are (lat, lon, 3rd dim)
* `time_coverage_start` and `time_coverage_end` exist as attributes in the files.
* `ds[ds_vec_name]` is a 1D coordinate aligned with the last dimension of ds_var_name.

Process Steps:

1.  Subset our Argo dataframe to only those rows within in the  time window covered by the cloud-hosted xarray dataset (`ds`). Let's call that `df_day`
2.  Get the lat/lon indices of `ds` that correspond to the lat/lon rows in `df_day`. These indices will look like `[0, 3, 6]` and `[0, 10, 15]` corresponding to index pairs, like `(3, 10)` corresponding to the lat/lon pair in a row of `df_day`.
3.  Do a fast point retrieval from `ds`. `ds` is cloud optimized (chunked) file and we only want to the chunks that we need. We do not need to load all the chunks. I/O of cloud-hosted data is the slow part so don't touch more of the data than you need to.
4. Build a dataframe with the predictor variables that we want from `ds`.

In [2]:
## The function to process one file
from typing import Optional, Tuple
import xarray as xr
import pandas as pd
import numpy as np

def one_file_matches(
    f: "earthaccess.store.EarthAccessFile",
    df: pd.DataFrame,
    ds_lat_name: str = "lat",
    ds_lon_name: str = "lon",
    ds_time_name: str = "time",
    ds_vec_name: Optional[str] = "wavelength",
    ds_var_name: str = "Rrs",
    ds_vec_sel = None,
    df_lat_name: str = "lat",
    df_lon_name: str = "lon",
    df_time_name: str = "time",
) -> Tuple[Optional[pd.DataFrame], Optional[pd.DataFrame]]:
    """
    Match Argo point observations to a single PACE L2/L3 file and extract
    colocated satellite values and metadata.

    Parameters
    ----------
    f : file-like object
        An earthaccess/open file-like handle for a single PACE granule
        (as returned by `earthaccess.open`). This object is passed directly
        to `xr.open_dataset` to read the granule.
    df : pandas.DataFrame
        A DataFrame containing Argo observations. Must include columns for
        time, latitude, longitude, and the target Argo variable to be matched.

    ds_lat_name, ds_lon_name, ds_time_name : str, optional
        Names of the latitude, longitude, and time variables in the PACE
        dataset. Default: "lat", "lon", "time".

    ds_vec_name : str or None, optional
        Name of the spectral dimension in the PACE dataset (e.g. "wavelength").
        If not None, matched satellite spectra are returned with one column per
        wavelength. If None, only a single variable is extracted.

    ds_vec_sel : value or None, optional
        Value of the spectral dimension in the PACE dataset (e.g. "wavelength")
        to select. If None, matched satellite spectra are returned with one column per
        wavelength. If given, only a single variable is extracted for that value.

    ds_var_name : str, optional
        Name of the satellite variable to extract from the PACE dataset
        (e.g. "Rrs" or "chlor_a").

    df_lat_name, df_lon_name, df_time_name : str, optional
        Column names in `df` for latitude, longitude, and time.

    df_var_name : str, optional
        Column name in `df` for the Argo variable being matched.

    Returns
    -------
    df_record : pandas.DataFrame or None
        A subset of `df` containing only the Argo observations whose timestamps
        fall within the PACE file's temporal coverage window. Returns None if
        no observations fall within this window.

    pts : pandas.DataFrame or None
        A DataFrame containing colocated satellite values for each matched Argo
        observation, including:
            - matched PACE pixel coordinates
            - the Argo variable value
            - spectral satellite values (if `ds_vec_name` is provided)
            - PACE temporal metadata (`pace_t_start`, `pace_t_end`)
            - the PACE file name (`pace_file`)
        Returns None if no points were matched.

Examples
    --------
    >>> import earthaccess
    >>> import pandas as pd
    >>>
    >>> # Log in and search for PACE granules (simplified example)
    >>> earthaccess.login()
    >>> results = earthaccess.search(
    ...     short_name="PACE_OCI_L3M_DAY_IOP",
    ...     temporal=("2024-03-05", "2024-03-06"),
    ...     bounding_box=(-180, -90, 180, 90),
    ... )
    >>> files = earthaccess.open(results, pqdm_kwargs={"disable": True})
    >>>
    >>> # Load Argo matchup candidates
    >>> df_argo = pd.read_parquet("tutorial_data/chl_argo_points.parquet")
    >>>
    >>> # Match a single PACE file to the Argo DataFrame
    >>> df_record, pts = one_file_matches(
    ...     files[0],
    ...     df_argo,
    ...     ds_vec_name=None,            # e.g. non-spectral variable like "chlor_a"
    ...     ds_var_name="chlor_a",
    ...     df_var_name="argo_chl"
    ... )
    
    Notes
    -----
    - Time matching uses a half-open interval: `[t_start, t_end)`.
    - Spatial matching is performed using nearest-neighbor lookup on the PACE
      latitude/longitude grid.
    - This function does not load full PACE granules into memory; only metadata
      and the required pixel values are accessed.
    - Returned DataFrames are aligned row-by-row: each row in `pts` corresponds
      to the same row in `df_record`.
    """
    with xr.open_dataset(f, chunks={}, cache=False) as ds:

        # --- time window in ds ---
        t_start = pd.to_datetime(ds.attrs["time_coverage_start"], utc=True)
        t_end   = pd.to_datetime(ds.attrs["time_coverage_end"], utc=True)

        # filename / product name
        fname = ds.attrs.get("product_name", None)
        if fname is None:
            src = ds.encoding.get("source", "")
            fname = src.split("/")[-1] if "/" in src else src

        df_times = pd.to_datetime(df[df_time_name], utc=True)
        # Use 24 hours so as not to cut off data if the window is small
        t_end_24 = t_start + pd.Timedelta(hours=24)
        df_record = df[(df_times >= t_start) & (df_times < t_end_24)]

        if df_record.empty:
            return None, None

        # --- spatial index / nearest lat-lon ---
        lat_idx = ds.get_index(ds_lat_name)
        lon_idx = ds.get_index(ds_lon_name)
        lat_vals = ds[ds_lat_name].values
        lon_vals = ds[ds_lon_name].values

        df_lats = df_record[df_lat_name].to_numpy(dtype=float)
        df_lons = df_record[df_lon_name].to_numpy(dtype=float)

        lat_i = lat_idx.get_indexer(df_lats, method="nearest")
        lon_i = lon_idx.get_indexer(df_lons, method="nearest")

        def sample_few_points(ds, lats, lons, var_name=ds_var_name):
            ds_var = ds[var_name]  # e.g. (lat, lon, wavelength)
            spectra = [
                ds_var.sel(lat=i, lon=j, method="nearest").values
                for i, j in zip(lats, lons)
            ]
            return np.stack(spectra, axis=0)

        var_vals = sample_few_points(ds, df_lats, df_lons)

        n = len(df_record)

        # --- build dataframe ---
        data = {
            # PACE file metadata per row
            f"pace_{ds_var_name}_file":  np.full(n, fname),
            f"pace_{ds_var_name}_t_start": np.full(n, t_start),
            f"pace_{ds_var_name}_t_end":   np.full(n, t_end),
            f"pace_{ds_var_name}_lat": lat_vals[lat_i],
            f"pace_{ds_var_name}_lon": lon_vals[lon_i],
        }

        if ds_vec_name is not None:
            vec_vals = ds[ds_vec_name].values
            if ds_vec_sel is not None:
                m = vec_vals == ds_vec_sel
                vec_vals = vec_vals[m]
            for j, v in enumerate(vec_vals):
                label = int(v)
                col_name = f"pace_{ds_var_name}_{label}"
                data[col_name] = var_vals[:, j].astype(float)
        else:
            col_name = f"pace_{ds_var_name}"
            data[col_name] = var_vals[:].astype(float)

        pts = pd.DataFrame(data)
        return df_record, pts

## Run through the PACE files in batches

Just in case something hangs.

### Workflow

* Create a function to get all matches for one PACE file `one_file_matches()`
* Create a function to run a batch of PACE files `run_batch()`
* Do a for loop over our batches and run `run_batch()` on each
* Gather results into one file

We will run for all PACE files that we have. Later when we match to the Argo profiles, we can choose the best PACE data, e.g. only use NRT data if that is all that is available.

### `run_batch()` function

* Run `one_file_matches()` on each file
* Save results from the batch to a file in case the process crashes
* Later we will reconstruct from the batch files. The files are small.

In [3]:
%%time
from datetime import datetime

def run_batch(
    results, df, 
    ds_vec_name="wavelength", ds_var_name="Rrs", ds_vec_sel=None,
    df_time_name="time", df_lat_name="lat", df_lon_name="lon"
):
    """
    Run a batch of PACE files (results) and return PACE variables for lat/lon/time rows in a
    dataframe.

    Parameters
    ----------
    results : earthaccess results object as returned by `earthaccess.search`). 
    
    df : A pandas DataFrame containing Argo observations. Must include columns for
        time, latitude, longitude.

    ds_vec_name : str or None, optional
        Name of the spectral dimension in the PACE dataset (e.g. "wavelength").
        If not None, matched satellite spectra are returned with one column per
        wavelength. If None, only a single variable is extracted.

    ds_vec_sel : value or None, optional
        Value of the spectral dimension in the PACE dataset (e.g. "wavelength")
        to select. If None, matched satellite spectra are returned with one column per
        wavelength. If given, only a single variable is extracted for that value.

    ds_var_name : str, optional
        Name of the variable to extract from the PACE dataset
        (e.g. "Rrs" or "chlor_a").

    df_lat_name, df_lon_name, df_time_name : str, optional
        Column names in `df` for latitude, longitude, and time.
    """

    # Make sure to refresh fileset to minimize the chance that the token expires before we are done
    # this for loop consumes about 6Gb of RAM
    fileset = earthaccess.open(results, pqdm_kwargs={"disable": True} );
    
    df_plus = []
    for i, f in enumerate(fileset):
        df_record, pts = one_file_matches(
            f, df, 
            ds_vec_name=ds_vec_name, ds_var_name=ds_var_name, ds_vec_sel=ds_vec_sel,
            df_time_name=df_time_name, df_lat_name=df_lat_name, df_lon_name=df_lon_name)
        if df_record is None or len(df_record) == 0: 
            print(f"Skipped day {i} no data in df")
            continue
        # error check
        if len(df_record) != len(pts):
            raise ValueError(f"Row mismatch: df_record={len(df_record)}, pts={len(pts)}")
    
        # left "concat": keep df_record rows, add pts columns by position
        df_record_plus = pd.concat(
            [ 
                df_record.reset_index(drop=True), 
                pts.reset_index(drop=True),
            ], axis=1,)
        df_plus.append(df_record_plus)
    if not len(df_plus) == 0:
        df_plus = pd.concat(df_plus, ignore_index=True)
    return df_plus

CPU times: user 9 μs, sys: 1 μs, total: 10 μs
Wall time: 14.1 μs


## Get the Rrs Level 3 PACE Data

In [4]:
import earthaccess
auth = earthaccess.login()
# are we authenticated?
if not auth.authenticated:
    # ask for credentials and persist them in a .netrc file
    auth.login(strategy="interactive", persist=True)
import xarray as xr
df = df_points
results = earthaccess.search_data(
    short_name = "PACE_OCI_L3M_RRS",
    temporal = (df.time.min(), df.time.max()),
    granule_name="*.DAY.*.4km.*"
)
print(f"{len(results)} days of PACE data")

620 days of PACE data


Check the format of individuals day data.

In [10]:
import xarray as xr
fileset = earthaccess.open(results[0:1], pqdm_kwargs={"disable": True} )
ds = xr.open_dataset(
        fileset[0],
        chunks={}
    )
ds

<xarray.Dataset> Size: 26GB
Dimensions:     (lat: 4320, lon: 8640, wavelength: 172, rgb: 3,
                 eightbitcolor: 256)
Coordinates:
  * lat         (lat) float32 17kB 89.98 89.94 89.9 ... -89.9 -89.94 -89.98
  * lon         (lon) float32 35kB -180.0 -179.9 -179.9 ... 179.9 179.9 180.0
  * wavelength  (wavelength) float64 1kB 346.0 348.0 351.0 ... 714.0 717.0 719.0
Dimensions without coordinates: rgb, eightbitcolor
Data variables:
    Rrs         (lat, lon, wavelength) float32 26GB dask.array<chunksize=(16, 1024, 8), meta=np.ndarray>
    palette     (rgb, eightbitcolor) uint8 768B dask.array<chunksize=(3, 256), meta=np.ndarray>
Attributes: (12/64)
    product_name:                      PACE_OCI.20240305.L3m.DAY.RRS.V3_1.Rrs...
    instrument:                        OCI
    title:                             OCI Level-3 Standard Mapped Image
    project:                           Ocean Biology Processing Group (NASA/G...
    platform:                          PACE
    source:                            satellite observations from OCI-PACE
    ...                                ...
    identifier_product_doi:            10.5067/PACE/OCI/L3M/RRS/3.1
    keywords:                          Earth Science > Oceans > Ocean Optics ...
    keywords_vocabulary:               NASA Global Change Master Directory (G...
    data_bins:                         3861548
    data_minimum:                      -0.009998
    data_maximum:                      0.092371605

In [12]:
%%time
# test that run_batch() works
df = df_points
res=run_batch(results[0:1], df)
res.head()

CPU times: user 1.46 s, sys: 218 ms, total: 1.68 s
Wall time: 4.43 s


,time,lat,lon,pace_Rrs_file,pace_Rrs_t_start,pace_Rrs_t_end,pace_Rrs_lat,pace_Rrs_lon,pace_Rrs_346,pace_Rrs_348,...,pace_Rrs_706,pace_Rrs_707,pace_Rrs_708,pace_Rrs_709,pace_Rrs_711,pace_Rrs_712,pace_Rrs_713,pace_Rrs_714,pace_Rrs_717,pace_Rrs_719
0,2024-03-05 11:37:00.000000000,68.045173,-5.118235,PACE_OCI.20240305.L3m.DAY.RRS.V3_1.Rrs.4km.nc,2024-03-05 00:08:58.225000+00:00,2024-03-06 02:07:24.895000+00:00,68.062500,-5.104161,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-03-05 02:08:10.000000000,25.111515,148.146488,PACE_OCI.20240305.L3m.DAY.RRS.V3_1.Rrs.4km.nc,2024-03-05 00:08:58.225000+00:00,2024-03-06 02:07:24.895000+00:00,25.104164,148.145844,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-03-05 08:16:05.000999936,0.673300,-165.453300,PACE_OCI.20240305.L3m.DAY.RRS.V3_1.Rrs.4km.nc,2024-03-05 00:08:58.225000+00:00,2024-03-06 02:07:24.895000+00:00,0.687497,-165.437500,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-03-05 15:35:00.000000000,39.151501,-66.574600,PACE_OCI.20240305.L3m.DAY.RRS.V3_1.Rrs.4km.nc,2024-03-05 00:08:58.225000+00:00,2024-03-06 02:07:24.895000+00:00,39.145832,-66.562500,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-03-05 01:16:05.000000000,29.082600,-17.213600,PACE_OCI.20240305.L3m.DAY.RRS.V3_1.Rrs.4km.nc,2024-03-05 00:08:58.225000+00:00,2024-03-06 02:07:24.895000+00:00,29.062498,-17.229162,0.006416,0.0069,...,0.000218,0.000218,0.000224,0.000214,0.000214,0.000196,0.000174,0.000166,0.00016,0.000214


### Run through all the files by batches

This takes about 4 hours for Rrs.

In [5]:
from pathlib import Path
from datetime import datetime
import pandas as pd

results_dir = Path("_temp_data")
results_dir.mkdir(exist_ok=True)

all_batches = []
batch_size = 10
var = "CHLA"
results = results
df = df_points
ds_var = "Rrs"

for batch_idx, i in enumerate(range(0, len(results), batch_size), start=1):
    batch = results[i:i+batch_size]
    print(f"Batch {batch_idx}: {len(batch)} files: {datetime.now():%H:%M:%S}")

    # If this batch was already done, skip it
    batch_path = results_dir / f"{var}_matchups_{ds_var}_batch_{batch_idx:03d}.parquet"
    if batch_path.exists():
        print(f"  -> Skipping batch {batch_idx}, found {batch_path}")
        continue

    df_batch = run_batch(batch, df)  # returns a DataFrame

    # Save this batch immediately
    df_batch.to_parquet(batch_path, index=False)
    print(f"  -> Saved {len(df_batch)} rows to {batch_path}")


Batch 1: 10 files: 18:29:30
  -> Saved 164 rows to _temp_data/CHLA_matchups_Rrs_batch_001.parquet
Batch 2: 10 files: 18:31:08
  -> Saved 175 rows to _temp_data/CHLA_matchups_Rrs_batch_002.parquet
Batch 3: 10 files: 18:32:50


KeyboardInterrupt: 

In [5]:
# When everything is done, you can merge all batches:
from pathlib import Path
import pandas as pd

results_dir = Path("_temp_data/matchups")
var = "CHLA"
ds_var = "Rrs"

batch_files = sorted(results_dir.glob(f"{var}_matchups_{ds_var}_batch_*.parquet"))
dfs = [pd.read_parquet(f) for f in batch_files]
df_all = pd.concat(dfs, ignore_index=True)
df_all.to_parquet(f"{results_dir}/{var}_argo_{ds_var}_all.parquet", index=False)


In [6]:
print(f"Total rows: {len(df_all)}")

Total rows: 18119
